<a href="https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a small Decision Tree for the Ranking Signal Analysis lane.

The goal is to learn whether a small set of observable search and traffic signals can identify pages that deserve ranking-opportunity review. A Decision Tree is appropriate because it can represent simple threshold-based relationships and is easy to inspect.

I prefer a small interpretable model over a more complex model because the purpose of this first capstone model is to understand whether the signals add useful decision support beyond my Week-4 baseline, not to maximize complexity.

In [9]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(file_path)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Shape: (11694072, 31)
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06


In [10]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Files downloaded successfully.")

Files downloaded successfully.


In [11]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

# Download March and April files
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

# Read only a small random sample from each month.
# This avoids loading millions of rows into memory.
query = """
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_avg_position,
        gsc_clicks
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position IS NOT NULL
    USING SAMPLE 20000 ROWS (reservoir, 42)
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_avg_position AS future_avg_position
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position IS NOT NULL
    USING SAMPLE 20000 ROWS (reservoir, 42)
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.gsc_impressions,
    m.gsc_avg_position,
    m.gsc_clicks,
    a.future_avg_position
FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
    AND m.content_hash_id = a.content_hash_id
"""

model_df = duckdb.sql(
    query,
    params=[march_path, april_path]
).df()

print("Model rows:", len(model_df))
print("Columns:", model_df.columns.tolist())

display(model_df.head(10))

Model rows: 408
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_avg_position', 'gsc_clicks', 'future_avg_position']


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,future_avg_position
0,client_3f0ce4d44fe94f3d,content_c49e1f7eb01337ee,89,4.202247,0,3.121212
1,client_23a62021009f63c4,content_06bc723a966f9c7a,21,15.238095,0,12.400000
2,client_08a6a72ff48e62c0,content_92c3aecd1dbca856,37,0.864865,0,0.000000
3,client_3f0ce4d44fe94f3d,content_0671c76e69df5b24,4,9.500000,0,1.200000
4,client_73cda7b4e4f265ea,content_d7fd85e913e3c108,231,5.991342,1,4.041667
5,client_73cda7b4e4f265ea,content_525f9a36ef00610a,596,2.548658,1,9.419355
6,client_3f0ce4d44fe94f3d,content_ecc1bd52a173afba,14,41.642857,0,3.333333
7,client_62f4a7e64f5e0096,content_bea6e7094c38fded,17,11.941176,1,4.545455
8,client_157ffe4d4a595515,content_d5db1a1c3b7c98f5,1,5.000000,0,2.000000
9,client_fef1a8f436438636,content_357972bf0916c869,21,4.714286,0,5.333333


In [12]:
import numpy as np
import pandas as pd

# Target:
# 1 = average position improved in April
# 0 = did not improve

model_df["target_improved"] = (
    model_df["future_avg_position"] < model_df["gsc_avg_position"]
).astype(int)

print("Target distribution:")
print(model_df["target_improved"].value_counts())

print("\nTarget percentages:")
print(model_df["target_improved"].value_counts(normalize=True))

display(model_df.head(10))

Target distribution:
target_improved
0    236
1    172
Name: count, dtype: int64

Target percentages:
target_improved
0    0.578431
1    0.421569
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,future_avg_position,target_improved
0,client_3f0ce4d44fe94f3d,content_c49e1f7eb01337ee,89,4.202247,0,3.121212,1
1,client_23a62021009f63c4,content_06bc723a966f9c7a,21,15.238095,0,12.400000,1
2,client_08a6a72ff48e62c0,content_92c3aecd1dbca856,37,0.864865,0,0.000000,1
3,client_3f0ce4d44fe94f3d,content_0671c76e69df5b24,4,9.500000,0,1.200000,1
4,client_73cda7b4e4f265ea,content_d7fd85e913e3c108,231,5.991342,1,4.041667,1
5,client_73cda7b4e4f265ea,content_525f9a36ef00610a,596,2.548658,1,9.419355,0
6,client_3f0ce4d44fe94f3d,content_ecc1bd52a173afba,14,41.642857,0,3.333333,1
7,client_62f4a7e64f5e0096,content_bea6e7094c38fded,17,11.941176,1,4.545455,1
8,client_157ffe4d4a595515,content_d5db1a1c3b7c98f5,1,5.000000,0,2.000000,1
9,client_fef1a8f436438636,content_357972bf0916c869,21,4.714286,0,5.333333,0


In [13]:
from sklearn.model_selection import train_test_split

feature_columns = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_clicks"
]

X = model_df[feature_columns].copy()
y = model_df["target_improved"].copy()

# Remove missing values
valid = X.notna().all(axis=1)

X = X.loc[valid]
y = y.loc[valid]

print("Rows used for modeling:", len(X))
print("Features:", feature_columns)

Rows used for modeling: 408
Features: ['gsc_impressions', 'gsc_avg_position', 'gsc_clicks']


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 306
Test rows: 102


In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
import pandas as pd

# -----------------------------
# 1. Baseline
# -----------------------------
# Baseline predicts the majority class from the training data

majority_class = y_train.mode()[0]

baseline_predictions = [majority_class] * len(y_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_precision = precision_score(
    y_test,
    baseline_predictions,
    zero_division=0
)
baseline_recall = recall_score(
    y_test,
    baseline_predictions,
    zero_division=0
)
baseline_f1 = f1_score(
    y_test,
    baseline_predictions,
    zero_division=0
)

# -----------------------------
# 2. Logistic Regression
# -----------------------------

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

model_accuracy = accuracy_score(y_test, model_predictions)
model_precision = precision_score(
    y_test,
    model_predictions,
    zero_division=0
)
model_recall = recall_score(
    y_test,
    model_predictions,
    zero_division=0
)
model_f1 = f1_score(
    y_test,
    model_predictions,
    zero_division=0
)

# -----------------------------
# 3. Compare
# -----------------------------

comparison = pd.DataFrame({
    "Method": [
        "Majority baseline",
        "Logistic Regression"
    ],
    "Accuracy": [
        baseline_accuracy,
        model_accuracy
    ],
    "Precision": [
        baseline_precision,
        model_precision
    ],
    "Recall": [
        baseline_recall,
        model_recall
    ],
    "F1": [
        baseline_f1,
        model_f1
    ]
})

display(comparison)

,Method,Accuracy,Precision,Recall,F1
0,Majority baseline,0.578431,0.00,0.000000,0.000000
1,Logistic Regression,0.637255,0.65,0.302326,0.412698


In [16]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0]
})

importance["absolute_coefficient"] = importance["coefficient"].abs()

importance = importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(importance)

,feature,coefficient,absolute_coefficient
2,gsc_clicks,0.198016,0.198016
1,gsc_avg_position,0.046561,0.046561
0,gsc_impressions,-0.001598,0.001598


In [17]:
results = X_test.copy()

results["actual"] = y_test
results["predicted"] = model_predictions

results["correct"] = (
    results["actual"] == results["predicted"]
)

print("Incorrect predictions:", (~results["correct"]).sum())
print("Total test rows:", len(results))

display(
    results[results["correct"] == False].head(10)
)

Incorrect predictions: 37
Total test rows: 102


,gsc_impressions,gsc_avg_position,gsc_clicks,actual,predicted,correct
283,156,20.897436,0,1,0,False
224,349,5.805158,0,1,0,False
206,34,8.147059,0,1,0,False
343,30,4.500000,0,1,0,False
382,4,8.250000,0,1,0,False
354,6,49.000000,0,0,1,False
271,6,8.833333,0,1,0,False
227,13,36.769231,0,0,1,False
393,9,3.333333,0,1,0,False
179,156,28.474359,0,0,1,False


In [18]:
display(comparison)

,Method,Accuracy,Precision,Recall,F1
0,Majority baseline,0.578431,0.00,0.000000,0.000000
1,Logistic Regression,0.637255,0.65,0.302326,0.412698


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used March 2026 observations as features and April 2026 average position as the future outcome. The data was split into 75% training and 25% test observations using a stratified split with a fixed random state. No April outcome field was used as a model feature.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

# Model vs baseline

I compared a majority-class baseline with Logistic Regression on the same held-out test set.

The majority baseline achieved 57.84% accuracy, while Logistic Regression achieved 63.73% accuracy. Logistic Regression also achieved 0.65 precision, 0.30 recall, and 0.41 F1.

The model therefore improved accuracy over the baseline in this experiment. However, recall remained relatively low, so the model missed a substantial portion of pages whose average position improved. These results are directional because the experiment uses a small sampled dataset.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Logistic Regression made 37 incorrect predictions out of 102 test observations.

The errors show that March search signals do not always predict whether a page's average position will improve in April. Some pages with relatively strong search visibility or good average position were predicted incorrectly, while other pages with weaker signals were also difficult to classify.

This suggests that impressions, clicks, and average position capture only part of the factors related to future ranking movement. The model is therefore better treated as decision support rather than proof of future ranking behavior.

The relatively low recall of 0.30 is an important limitation because many improving pages were not identified by the model.

### Limitation

This experiment was run on a small sampled subset of the warehouse because the full monthly files exceeded the available Colab memory. Therefore, the reported metrics should be treated as directional and not as a final estimate of production performance.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.